Sprint 1: Importação dos dados

In [2]:
import pandas as pd # Importa a biblioteca Pandas para manipulação de tabelas de dados
import numpy as np  # Importa o NumPy para lidar com valores nulos matemáticos (NaN)
import re           # Importa a biblioteca de Expressões Regulares para limpeza avançada de textos


In [3]:
# Carrega a base de dados. O parâmetro sep=';' é crucial porque o arquivo original 
# não é separado por vírgulas, evitando que todos os dados fiquem em uma única coluna.
df = pd.read_csv('varejo.csv', sep=';')


In [4]:
print("--- Visão Inicial da Base (Sprint 1) ---")
display(df.info()) # Exibe a contagem de registros não nulos e os tipos de dados originais


--- Visão Inicial da Base (Sprint 1) ---
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


None

In [5]:
# Existem colunas vazias de "lixo" geradas por pontos e vírgulas excedentes no final de cada linha do CSV.
# A lógica '~' inverte a seleção, mantendo apenas as colunas que NÃO começam com 'Unnamed'.
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [6]:
print("--- Visão Após Correção da colunas fantasmas(Sprint 1) ---")
display(df.info()) # Exibe a contagem de registros não nulos e os tipos de dados sem as colunas fantasmanas
print("--- Visão da tabela com os campos e dados das 5 primeiras linhas (Sprint 1) ---")
display(df.head(5)) # Mostra as 5 primeiras linhas para conferência visual


--- Visão Após Correção da colunas fantasmas(Sprint 1) ---
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   DATA       830000 non-null  str  
 1   CO_ID      830000 non-null  int64
 2   CL_ID      830000 non-null  int64
 3   CL_GENERO  830000 non-null  str  
 4   CL_EC      830000 non-null  int64
 5   CL_FHL     830000 non-null  int64
 6   CL_SEG     830000 non-null  str  
 7   PR_ID      830000 non-null  int64
 8   PR_CAT     830000 non-null  str  
 9   PR_NOME    830000 non-null  str  
dtypes: int64(5), str(5)
memory usage: 63.3 MB


None

--- Visão da tabela com os campos e dados das 5 primeiras linhas (Sprint 1) ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


Sprint 2: Transformação de Tipos e Limpeza de Strings

In [7]:
# Bloco 1: Limpeza de texto em colunas de strings
if 'PR_NOME' in df.columns:
    # Usa expressão regular (re.sub) para encontrar espaços duplos ou múltiplos (\s+) 
    # e substituí-los por um único espaço, além de remover espaços nas bordas com strip().
    df['PR_NOME'] = df['PR_NOME'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()))
 

In [8]:
# Bloco 2: Transformação de Tipos Numéricos (Regra de Negócio)
if 'CO_ID' in df.columns:
    # Converte o ID da compra para número. O parâmetro errors='coerce' força 
    # qualquer texto inválido a virar um valor nulo (NaN), facilitando a exclusão depois.
    df['CO_ID'] = pd.to_numeric(df['CO_ID'], errors='coerce')

In [9]:
# Bloco 3: Transformação para Datetime
if 'DATA' in df.columns:
    # Converte a string de data para o formato datetime do Pandas. 
    # dayfirst=True garante que dias e meses não sejam invertidos no formato brasileiro.
    df['DATA'] = pd.to_datetime(df['DATA'], errors='coerce', dayfirst=True)


In [18]:
print("--- Tipos de Dados Após Transformação (Sprint 2) ---")
display(df.dtypes) # Comprova que as conversões de string para int/float/datetime funcionaram


--- Tipos de Dados Após Transformação (Sprint 2) ---


DATA         datetime64[us]
CO_ID                 int64
CL_ID                 int64
CL_GENERO               str
CL_EC                 int64
CL_FHL                int64
CL_SEG                  str
PR_ID                 int64
PR_CAT                  str
PR_NOME                 str
dtype: object

Sprint 3: Limpeza de Nulos e Duplicatas

In [11]:
print("--- Diagnóstico de Problemas ---")
print(f"Total de duplicatas encontradas: {df.duplicated().sum()}") # Conta linhas idênticas
print(f"Total de valores nulos encontrados: {df.isnull().sum().sum()}") # Conta valores nulos em todas as colunas

--- Diagnóstico de Problemas ---
Total de duplicatas encontradas: 96553
Total de valores nulos encontrados: 0


In [12]:
# ==========================================
# INVESTIGAÇÃO DE DUPLICATAS (Opcional para a Sprint 3)
# ==========================================
# O parâmetro keep=False faz o pandas trazer TODAS as cópias das linhas idênticas
duplicatas = df[df.duplicated(keep=False)]

# Ordenamos pelo ID da Compra e ID do Produto para os "clones" ficarem lado a lado
if 'CO_ID' in df.columns and 'PR_ID' in df.columns:
    duplicatas = duplicatas.sort_values(by=['CO_ID', 'PR_ID'])

print("--- Amostra das Linhas Duplicadas ---")
display(duplicatas.head(10)) # Mostra os 10 primeiros clones

--- Amostra das Linhas Duplicadas ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
40,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
7,2019-02-01,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE
46,2019-02-01,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE
14,2019-02-01,1000,534,M,4,1,C,13,ALIMENTOS,BANANA
19,2019-02-01,1000,534,M,4,1,C,13,ALIMENTOS,BANANA
22,2019-02-01,1000,534,M,4,1,C,69,BEBIDAS,REFRIGERANTE LIMaO
49,2019-02-01,1000,534,M,4,1,C,69,BEBIDAS,REFRIGERANTE LIMaO
15,2019-02-01,1000,534,M,4,1,C,218,ALIMENTOS,BIFE DE COXAO MOLE
50,2019-02-01,1000,534,M,4,1,C,218,ALIMENTOS,BIFE DE COXAO MOLE


In [13]:
if 'PR_CAT' in df.columns:
    print(f"Problema 2: Inconsistência na categoria com {(df['PR_CAT'] == '#N/D').sum()} valores '#N/D'.")
print(f"Problema 3: Valores nulos gerais na base:\n{df.isnull().sum()}\n")

Problema 2: Inconsistência na categoria com 3650 valores '#N/D'.
Problema 3: Valores nulos gerais na base:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64



In [14]:
print("--- EXECUTANDO LIMPEZA MÍNIMA NECESSÁRIA (ETAPA OBRIGATÓRIA 3) ---")
# 1. Eliminar duplicatas relevantes
# df.drop_duplicates(inplace=True)

# 2. Imputar nulos com justificativa
if 'PR_CAT' in df.columns:
    df['PR_CAT'] = df['PR_CAT'].replace('#N/D', np.nan)
    # Justificativa 1: Preencher com 'Sem Categoria' preserva a linha para não perder o volume de compras do cliente.
    df['PR_CAT'] = df['PR_CAT'].fillna('Sem Categoria')

--- EXECUTANDO LIMPEZA MÍNIMA NECESSÁRIA (ETAPA OBRIGATÓRIA 3) ---


In [15]:
# Exclui permanentemente as linhas onde o ID de compra ficou inválido na Sprint 2
df = df.dropna(subset=['CO_ID'])

print("\n--- Base Após Limpeza (Sprint 3) ---")
print(f"Nulos restantes:\n{df.isnull().sum()}") # Mostra que a base agora está limpa


--- Base Após Limpeza (Sprint 3) ---
Nulos restantes:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64



SPRINT 4: Estatística Descritiva e Padrões de Agrupamento

In [19]:
# ETAPA OBRIGATÓRIA 4: Estatística de número de filhos
if 'CL_FHL' in df.columns:
    print("--- ESTATÍSTICAS DESCRITIVAS: NÚMERO DE FILHOS ---")
    # O método .describe() retorna float64 por padrão porque inclui média (mean) e desvio padrão (std).
    # Usamos .round(0).astype(int) para arredondar e converter toda a série para inteiro (int64).
    estatisticas = df['CL_FHL'].describe().round(0).astype(int)
    moda = df['CL_FHL'].mode()[0]
    print(estatisticas)
    print(f"Moda: {moda}\n")

--- ESTATÍSTICAS DESCRITIVAS: NÚMERO DE FILHOS ---
count    830000
mean          1
std           1
min           0
25%           0
50%           0
75%           2
max           4
Name: CL_FHL, dtype: int64
Moda: 0



In [17]:
# ETAPA OBRIGATÓRIA 5: Padrões de Agrupamento
print("--- PADRÕES DE AGRUPAMENTO ---")
print("Agrupamento 1 (Quais categorias vendem mais):")
if 'PR_CAT' in df.columns:
    display(df.groupby('PR_CAT')['PR_ID'].count().sort_values(ascending=False))

print("\nAgrupamento 2 (Quem compra mais - Média de filhos por gênero):")
if 'CL_GENERO' in df.columns and 'CL_FHL' in df.columns:
    display(df.groupby('CL_GENERO')['CL_FHL'].mean())

--- PADRÕES DE AGRUPAMENTO ---
Agrupamento 1 (Quais categorias vendem mais):


PR_CAT
ALIMENTOS        434767
HIGIENE          155574
LIMPEZA          145754
BEBIDAS           43299
PET               32399
ACESSORIOS        14557
Sem Categoria      3650
Name: PR_ID, dtype: int64


Agrupamento 2 (Quem compra mais - Média de filhos por gênero):


CL_GENERO
F    1.090659
M    1.207363
Name: CL_FHL, dtype: float64